# Suite COL — Collections

A collection is a list (mbse-schemas' `OfIndexed`, positional or keyed) or an object's entries in an adjacency. Basic
reads a list property as a `Domains.Collection` and an object's entries as records, counts, indexes, tests membership,
adds, finds the least and the greatest and tests uniqueness, and binds a name to each item with the quantifiers `all`,
`any` and `count`, all by Kleene's logic. Collections have domains for inference only: `Domains.List` and
`Domains.Keyed`.

In [ ]:
import { Domains as D, Evaluators as V, Expressions as E } from "@mbse/expressions";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import { Domains as FD, Errors as FE, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import * as T from "@mbse/expressions/Translators";
import { Errors, JSON, Proxies, Schemas as S } from "@mbse/schemas/Framework";
import { register } from "@mbse/expressions";

const store = register(new Proxies.OfStore()); // holds every dialect's meta-schemas

const { NotImplementedError, ValueError } = Errors;
const { OverflowError } = FE;
const L = Latex.Expressions;
import { assert, equal, raises, same, text } from "./support.js";

const listed = (spec: any) => (t: any) => t.as_indexed((i: any) => i.of(spec));
const [Int, Float, Str] = [BigInt, Number, String].map((t) => new S.OfNative.Data(t));
const Port = new S.OfObject.Builder().ref().properties(text("name"), text("width", BigInt)).create();
const Wire = new S.OfRelation.Builder().links("component", "port").properties(text("pin", BigInt)).create();
const Component = new S.OfObject.Builder().ref().properties(
  text("name"), (p) => p.name("sizes").of(listed(Int)), (p) => p.name("grid").of(listed(listed(Int))),
  (p) => p.name("limits").of((t) => t.as_indexed((i) => i.key(Str).of(Float))),
  (p) => p.name("labels").of(listed(Str))).relations((a) => a.name("ports").of(Wire).me("component")).create();
for (const [name, schema] of [["Port", Port], ["Wire", Wire], ["Component", Component]] as const) store.register(name, schema);
const B = store as any;
const a = B.Port().name("a").width(8n).create();
const b = B.Port().name("b").width(16n).create();
const amp = B.Component().name("amp").sizes([3n, 1n, 2n]).grid([[1n, 2n], [3n]]).limits(new Map([["lo", 0.5], ["hi", 2.0]]))
  .labels(["x", "y", "x"]).ports((w: any) => w.port(a).pin(1n)).ports((w: any) => w.port(b).pin(2n)).create();
const [self, x, xs] = [E.variable("this"), E.variable("x"), E.variable("xs")];

/** The value of a Basic expression, with `this` bound to `amp` unless given. */
function ev(expression: unknown, variables: Record<string, unknown> = {}): any {
  return V.OfAny(expression as never, { this: amp, ...variables });
}

const collection = (items: unknown[], keys: unknown[] | null = null) => new D.Collection(items, keys);

## COL-01 · A list property reads as a collection, positional or keyed, nested lists too; a scope's list is positional

In [ ]:
{
  const sizes = ev(self.sizes);
  assert(collection([3n, 1n, 2n]).equals(sizes) && sizes.keys === null && String(sizes) === "[3, 1, 2]");
  assert(collection([collection([1n, 2n]), collection([3n])]).equals(ev(self.grid)));
  const limits = ev(self.limits);
  assert(collection([0.5, 2.0], ["lo", "hi"]).equals(limits) && String(limits) === "{'lo': 0.5, 'hi': 2.0}");
  assert(ev(self.get("missing")) === null && ev(self.has("sizes")) === true);
  assert(ev(xs.count(), { xs: [1n, 2n] }) === 2n && ev(xs.count(), { xs: [1n, 2n, 3n] }) === 3n && ev(xs.count(), { xs: collection([]) }) === 0n);
  const I8 = new D.OfInteger.Data(8n);
  const ab = new TextEncoder().encode("ab");
  assert(collection([ab, new D.Value(I8, 1n)]).equals(collection([new TextEncoder().encode("ab"), new D.Value(new D.OfInteger.Data(8n), 1n)])));
  assert(!collection([ab]).equals(collection([new TextEncoder().encode("ac")])) && !collection([1n], ["k"]).equals(collection([1n])));
  // Collections compare equal to nothing: eq is unknown.
  assert(ev(self.sizes.eq(self.sizes)) === null);
}

## COL-02 · count, item and in: positions from 0 and keys, unknown where there is no item

In [ ]:
{
  assert(ev(self.sizes.count()) === 3n && ev(self.limits.count()) === 2n);
  assert(ev(self.sizes.item(0n)) === 3n && ev(self.sizes.item(2n)) === 2n && ev(self.sizes.item(3n)) === null);
  assert(ev(self.sizes.item(-1n)) === null && ev(self.sizes.item(E.literal(1n, new D.OfInteger.Data(8n)))) === 1n);
  assert(ev(self.limits.item("hi")) === 2.0 && ev(self.limits.item("mid")) === null && ev(self.grid.item(1n).item(0n)) === 3n);
  assert(ev(E.literal(2n).in_(self.sizes)) === true && ev(E.literal(5n).in_(self.sizes)) === false);
  assert(ev(E.literal(2.0).in_(self.limits)) === true && ev(E.literal(1.0).in_(self.limits)) === false);
  assert(ev(E.literal("lo").in_(self.limits)) === null); // items, not keys: a str is incomparable with them
  assert(ev(E.literal(2.0).in_(self.sizes)) === null); // incomparable with every item: unknown
  assert(ev(E.literal(1n).in_(xs), { xs: [1.0, 1n] }) === true && ev(E.literal(2n).in_(xs), { xs: [1.0, 1n] }) === null);
  for (const unknown of [self.get("nope").count(), self.get("nope").item(0n), self.sizes.item(self.get("nope")),
    E.literal(1n).in_(self.get("nope")), self.get("nope").in_(self.sizes)]) assert(ev(unknown) === null);
  raises(TypeError, () => ev(E.literal(1n).count()), "count expects a collection, got int");
  raises(TypeError, () => ev(self.sizes.item("0")), "item expects an integer position, got str");
}

## COL-03 · sum, min, max and unique, by the domains of the items

In [ ]:
{
  assert(ev(self.sizes.sum()) === 6n && ev(self.limits.sum()) === 2.5 && ev(xs.sum(), { xs: [] }) === 0n);
  const I8 = new D.OfInteger.Data(8n);
  assert(new D.Value(I8, 127n).equals(ev(xs.sum(), { xs: [new D.Value(I8, 100n), new D.Value(I8, 27n)] })));
  raises(OverflowError, () => ev(xs.sum(), { xs: [new D.Value(I8, 100n), new D.Value(I8, 28n)] }), "sum overflows int8: 128");
  raises(TypeError, () => ev(xs.sum(), { xs: [1n, 2.0] }), "sum expects numbers of one domain, got int and float");
  raises(TypeError, () => ev(self.labels.sum()), "sum expects numbers, got str");
  assert(ev(self.sizes.min()) === 1n && ev(self.sizes.max()) === 3n && ev(self.labels.max()) === "y");
  assert(ev(xs.min(), { xs: [] }) === null && ev(xs.max(), { xs: [1.0, NaN] }) === null); // incomparable
  assert(new D.Value(I8, -5n).equals(ev(xs.min(), { xs: [new D.Value(I8, 5n), new D.Value(I8, -5n)] })));
  raises(TypeError, () => ev(xs.min(), { xs: [1n, "a"] }), "min expects ordered values of one domain, got int and str");
  raises(TypeError, () => ev(xs.max(), { xs: [true, false] }), "max expects ordered values of one domain, got bool and bool");
  assert(ev(self.sizes.unique()) === true && ev(self.labels.unique()) === false && ev(xs.unique(), { xs: [1n, 1.0] }) === null);
  assert(ev(xs.unique(), { xs: [] }) === true);
  for (const unknown of [self.get("nope").sum(), self.get("nope").min(), self.get("nope").unique()]) assert(ev(unknown) === null);
  assert(ev(xs.sum(), { xs: [1n, null] }) === null && ev(xs.max(), { xs: [1n, null] }) === null);
}

## COL-04 · entries gives an object's entries in an adjacency as records, which get and has read

In [ ]:
{
  const ports = ev(self.entries("ports"));
  assert(ports.items.length === 2 && ports.keys === null);
  const first = ports.items[0];
  assert(first instanceof D.Record && equal(first.fields, new Map<string, unknown>([["port", a], ["pin", 1n]])));
  assert(String(first).startsWith("record(port="));
  assert(ev(self.entries("ports").item(1n).get("pin")) === 2n && ev(self.entries("ports").item(0n).has("pin")) === true);
  assert(ev(self.entries("ports").item(0n).get("port").eq(E.variable("a")), { a }) === true); // the linked object itself
  assert(ev(self.entries("ports").item(0n).get("port").get("width")) === 8n && ev(self.entries("nope").count()) === 0n);
  assert(ev(E.variable("o").entries("ports"), { o: null }) === null);
  raises(TypeError, () => ev(self.entries(1n as never)), "entries expects an adjacency name, got int");
  raises(TypeError, () => ev(E.literal(1n).entries("ports")), "entries expects an object, got int");
  raises(TypeError, () => ev(self.sizes.get("x")), "get expects an object, got Collection");
}

## COL-05 · Quantifiers bind a name to each item: all and any by Kleene's logic, count of the items the body holds for

In [ ]:
{
  assert(ev(self.sizes.all("x", x.gt(0n))) === true && ev(self.sizes.all("x", x.gt(1n))) === false);
  assert(ev(self.sizes.any("x", x.gt(2n))) === true && ev(self.sizes.any("x", x.gt(5n))) === false);
  assert(ev(self.sizes.count_where("x", x.gt(1n))) === 2n && ev(self.limits.all("x", x.gt(0.0))) === true);
  assert(ev(xs.all("x", x.gt(0n)), { xs: [] }) === true && ev(xs.any("x", x.gt(0n)), { xs: [] }) === false);
  assert(ev(xs.count_where("x", true), { xs: [] }) === 0n);
  // Unknown bodies: false or true decide, otherwise unknown; count is unknown.
  assert(ev(xs.all("x", x.gt(0n)), { xs: [1n, "a"] }) === null && ev(xs.all("x", x.gt(0n)), { xs: ["a", 0n] }) === false);
  assert(ev(xs.any("x", x.gt(0n)), { xs: ["a", 0n] }) === null && ev(xs.any("x", x.gt(0n)), { xs: ["a", 1n] }) === true);
  assert(ev(xs.count_where("x", x.gt(0n)), { xs: [1n, "a"] }) === null);
  assert(ev(self.get("nope").all("x", true)) === null);
  // Over entries: every port's width is at least 8, and the pins are unique.
  assert(ev(self.entries("ports").all("e", E.variable("e").get("port").get("width").ge(8n))) === true);
  assert(ev(self.entries("ports").count_where("e", E.variable("e").get("pin").gt(1n))) === 1n);
  // Nested, and the name shadows an outer binding within the body only.
  assert(ev(self.grid.all("row", E.variable("row").any("x", x.ge(3n)))) === false);
  assert(ev(E.let_("x", 10n, self.sizes.all("x", x.lt(10n)).and_(x.eq(10n)))) === true);
  raises(TypeError, () => ev(self.sizes.all("x", x)), "all expects bool operands, got int");
  raises(TypeError, () => ev(E.literal(1n).any("x", true)), "any expects a collection, got int");
  assert(V.OfQuantifier(E.quantifier("all", "x", xs, x.ge(1n)), { xs: [1n, 2n] }) === true);
}

## COL-06 · The quantifier kind: built, validated as a binding, stored and read back, and translated nowhere yet

In [ ]:
{
  const built = new E.OfQuantifier.Builder().name("x").quantifier("any").collection(xs).body(x.gt(1n)).create();
  const form = built.form();
  assert(form.kind === "quantifier" && form.attributes.size === 2 && form.attributes.get("quantifier") === "any");
  assert(form.arguments[0] === xs.data && form.arguments[1] === built.body);
  assert(F.same(E.DIALECT.make(form), built) && F.same(E.OfAny.resolve((b: any) => b.as_quantifier(built)), built));
  assert(built.validate({ bound: ["xs"] }).length === 0 && same([...SY.free(built)], ["xs"]));
  assert(same(built.validate(), ["collection: variable 'xs' is not bound"]));
  assert(same(E.quantifier("all", "x", xs, E.variable("y")).data.validate({ bound: ["xs"] }), ["body: variable 'y' is not bound"]));
  assert(same(E.quantifier("every", "x", xs, true).data.validate({ bound: ["xs"], core: true }), ["'every' is not a core operation"]));
  assert(same(new E.OfQuantifier.Data("x", "all", xs.data).validate({ bound: ["xs"] }), ["a quantifier needs a body"]));
  assert(same(new E.OfQuantifier.Data(null, "all", xs.data, E.literal(true).data).validate({ bound: ["xs"] }), ["a quantifier needs a name"]));
  raises(ValueError, () => V.OfAny(new E.OfQuantifier.Data("x", "all", xs.data), { xs: [] }), "a quantifier needs a body");
  raises(NotImplementedError, () => V.OfAny(E.quantifier("every", "x", xs, true), { xs: [] }), "'every' is not a core operation");
  const text_ = JSON.ToJSON(store).Reachable(E.OfQuantifier.Schema, built);
  const back = JSON.FromJSON(E.Builders).Reachable(E.OfQuantifier.Schema, text_);
  assert(F.same(back, built) && E.Builders.name_of(E.OfQuantifier.Schema) === "Expressions.OfQuantifier");
  raises(ValueError, () => T.between(E.DIALECT, L.DIALECT).forward(self.sizes.all("x", x.gt(0n))),
    "Basic quantifier 'all' has no Latex counterpart");
}

## COL-07 · Collections have domains for inference: a quantifier's name has the items' domain

In [ ]:
{
  const [ints, keyed] = [new D.List(D.Int), new D.Keyed(D.Str, D.Float)];
  assert(ints.name() === "list(int)" && String(ints) === "list(int)" && String(keyed) === "keyed(str, float)");
  assert(ints.equals(new D.List(D.Int)) && !ints.equals(new D.List(D.Str)));
  assert(new D.List(new D.OfInteger.Data(8n)).equals(new D.List(new D.OfInteger.Data(8n))));
  assert(ints.contains(collection([1n, 2n])) && ints.contains([1n]) && !ints.contains(collection([1n], ["a"])));
  assert(!ints.contains([1.5]) && keyed.contains(collection([1.5], ["a"])) && !keyed.contains(collection([1.5])));
  assert(ints.includes(new D.List(D.Int)) && !ints.includes(new D.List(D.Str)) && ints.includes(new FD.OfUnion(ints)));
  assert(keyed.includes(new D.Keyed(D.Str, D.Float)) && keyed.includes(new FD.OfUnion(keyed)) && !keyed.includes(ints));
  assert(D.items_of(ints) === D.Int && D.items_of(keyed) === D.Float && D.items_of(D.Int) === FD.Anything);
  assert(D.Records.contains(new D.Record(new Map())) && !D.Records.contains(amp));
  const infer = (expression: unknown, environment?: Record<string, FD.Domain>) => E.DIALECT.infer(expression, environment);
  assert(infer(xs.all("x", x.gt(0n)), { xs: ints }) === D.Bool && infer(xs.count_where("x", x.gt(0n)), { xs: ints }) === D.Int);
  assert(infer(xs.sum(), { xs: new D.List(D.Float) }) === D.Float && infer(xs.item(0n), { xs: keyed }) === D.Float);
  assert(infer(xs.min(), { xs: new D.List(D.Str) }) === D.Str && infer(xs.count(), { xs: FD.Anything }) === D.Int);
  assert(infer(E.literal(1n).in_(xs), { xs: ints }) === D.Bool && infer(xs.unique(), { xs: ints }) === D.Bool);
  assert(new D.List(D.Records).equals(infer(self.entries("ports"), { this: D.Object })));
  assert(infer(self.entries("ports").all("e", E.variable("e").get("pin").gt(1n)), { this: D.Object }) === D.Bool);
  assert(infer(E.quantifier("every", "x", xs, x), { xs: ints }) === FD.Anything); // an extension: its name is unknown too
  raises(TypeError, () => infer(xs.sum(), { xs: new D.List(D.Str) }), "sum cannot take (list(str)); it takes (collection of number) -> number");
  raises(TypeError, () => infer(E.literal(1n).count()), "count cannot take (int); it takes (collection of T) -> int");
  raises(TypeError, () => infer(E.literal(1n).in_(2n)), "in cannot take (int, int); it takes (T, collection of T) -> bool");
  raises(TypeError, () => infer(xs.all("x", x), { xs: ints }), "all cannot take (list(int), int); it takes (collection, bool) -> bool");
  raises(TypeError, () => infer(E.literal(1n).any("x", true)), "any cannot take (int, bool); it takes (collection, bool) -> bool");
}